# Notebook 2 — Statistical Analysis & Anomaly Detection

## Objective
Run OLS regression to identify what drives uptake gaps, compute residuals, and identify the 10 districts with anomalously low uptake *after controlling for observable characteristics*.

## Methodology
We estimate:

> `uptake_rate = β₀ + β₁·literacy + β₂·female_literacy + β₃·rural_share + β₄·sc_st_share + β₅·internet + ε`

Districts with large **negative residuals** (ε << 0) are those performing worse than their socioeconomic profile predicts — these are our intervention targets.

**Limitation:** This analysis identifies correlations. Causal interpretation requires experimental design beyond the scope of this study.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.= __import__('warnings'); warnings.filterwarnings('ignore')

df = pd.read_csv('../data/clean/pmkisan_clean.csv')
print(f"Loaded {len(df)} clean districts")
df[['uptake_rate','literacy_rate','female_literacy','sc_st_share','internet_access']].describe().round(3)


In [ ]:
# OLS Regression
X_cols = ['literacy_rate', 'female_literacy', 'rural_share', 'sc_st_share', 'internet_access']
df_model = df.dropna(subset=X_cols + ['uptake_rate'])
X = sm.add_constant(df_model[X_cols])
y = df_model['uptake_rate']

model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
# Extract residuals and flag anomalies
df.loc[df_model.index, 'residual'] = model.resid
df.loc[df_model.index, 'predicted_uptake'] = model.fittedvalues
df['residual'] = df['residual'].fillna(0)

threshold = df['residual'].quantile(0.10)
df['is_anomaly'] = df['residual'] < threshold

print(f"Anomaly threshold (10th percentile residual): {threshold:.4f}")
print(f"Number of anomalous districts: {df['is_anomaly'].sum()}")

anomalies = df[df['is_anomaly']].sort_values('residual').head(10)
print("\nBottom 10 districts:")
print(anomalies[['state','district','uptake_rate','gap_rate','residual']].to_string())


In [ ]:
# Forest plot — regression coefficients
fig, ax = plt.subplots(figsize=(8, 5))
params = model.params.drop('const')
conf = model.conf_int().drop('const')
pvals = model.pvalues.drop('const')
labels = {'literacy_rate':'Literacy rate','female_literacy':'Female literacy',
          'rural_share':'Rural share','sc_st_share':'SC/ST share','internet_access':'Internet access'}

for i, (name, coef) in enumerate(params.items()):
    lo, hi = conf.loc[name]
    sig = pvals[name] < 0.05
    color = '#e05a2b' if (coef < 0 and sig) else ('#2e8b57' if (coef > 0 and sig) else '#888')
    ax.barh(i, coef, color=color, alpha=0.7, height=0.4)
    ax.plot([lo, hi], [i, i], color=color, linewidth=2.5)
    ax.plot([lo]*2, [i-0.12, i+0.12], color=color, linewidth=1.5)
    ax.plot([hi]*2, [i-0.12, i+0.12], color=color, linewidth=1.5)
    if sig: ax.text(max(hi, coef)+0.003, i, '★', va='center', color=color, fontsize=12)

ax.set_yticks(range(len(params)))
ax.set_yticklabels([labels[n] for n in params.index])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Regression coefficients with 95% CI  (R²={model.rsquared:.2f})\n★ = p<0.05', fontsize=11)
ax.set_xlabel('Effect on uptake rate')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()


In [ ]:
# Save analysis-ready dataset
df.to_csv('../data/clean/analysis_ready.csv', index=False)
anomalies.to_csv('../data/clean/bottom10_anomalies.csv', index=False)
print("Saved analysis_ready.csv and bottom10_anomalies.csv ✓")
print(f"\nKey finding: SC/ST share is the strongest significant predictor")
print(f"  Coefficient: {model.params['sc_st_share']:.4f}  p={model.pvalues['sc_st_share']:.4f}")
